# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '/content/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]

print('pop-tart-looking rows:', len(pop))
pop

pop-tart-looking rows: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
straw = pop[pop['product_name'].str.lower().str.contains('straw', na=False)]
not_straw = pop[~pop.index.isin(straw.index)]

print('strawberry Pop-Tart rows:', len(straw))
print(straw[['sku', 'product_name']].to_string(index=False))
print()
print('Pop-Tart rows that are NOT strawberry:', len(not_straw))
print(not_straw[['sku', 'product_name']].to_string(index=False))

strawberry Pop-Tart rows: 4
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Pop-Tart rows that are NOT strawberry: 0
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** none, all 4 Pop-Tart rows in the catalog are strawberry (though PT-12 is a 4-pack and POPTART-STRAW is a 6 pack).

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
CANON = {
    'PT-12': 'POPTART-STRAW', 'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW', 'PT_STRAWBERRY': 'POPTART-STRAW',
    'LTBEER24': 'BEER-24',
    'BATT_AA_4PK': 'BAT-AA4',
    'FIRSTAID-KIT': 'FAID-01',
    'BG-MONO': 'MONOPOLY-01', 'GAME-MONO': 'MONOPOLY-01',
    'WHITEBREAD': 'BREAD-WH',
    'PNUTBTR-16': 'PB-16',
    'CANDLE-EMRG': 'CNDL-EM',
    'CNSOUP-01': 'SOUP-CAN',
    'PLAYCARD-STD': 'CARDS-01',
    'CHIPVARIETY': 'CHIPS-VP',
    'HDTARP-01': 'TARP-HD',
    'LED-FLASH': 'FLASH-01',
    'PAPERTOWEL6': 'PTOWEL-6',
}

cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])

assert cat['canonical_sku'].notna().all(), 'mapping created NaN canonical SKUs'
print('rows:', len(cat), '| distinct SKUs before:', cat['sku'].nunique(),
      '| after:', cat['canonical_sku'].nunique())
cat

rows: 45 | distinct SKUs before: 45 | after: 28


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BAT-AA4


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print('before:')
print(cat['category'].value_counts().to_string())

cat['category'] = cat['category'].str.strip().str.title()
print('\nafter strip + title-case:')
print(cat['category'].value_counts().to_string())

CATEGORY_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Groceries': 'Grocery',
    'Baby Products': 'Baby',
    'Alcoholic Beverages': 'Alcohol',
}
cat['category'] = cat['category'].map(CATEGORY_MAP).fillna(cat['category'])
print('\nafter mapping:')
print(cat['category'].value_counts().to_string())

before:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1

after strip + title-case:
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods       

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
import re

def name_key(name):
    """lowercase, strip punctuation, sort the words"""
    words = re.sub(r'[^a-z0-9\s]', ' ', str(name).lower()).split()
    return ' '.join(sorted(words))

cat['name_key'] = cat['product_name'].apply(name_key)

groups = (cat.groupby('name_key')
             .agg(n_skus=('sku', 'nunique'),
                  skus=('sku', lambda s: sorted(set(s))),
                  names=('product_name', lambda s: sorted(set(s))))
             .reset_index())

candidates = groups[groups['n_skus'] > 1]
print('name keys:', len(groups), '| keys with more than one SKU:', len(candidates))
candidates

name keys: 29 | keys with more than one SKU: 14


,name_key,n_skus,skus,names
2,24 beer light pack,2,"[BEER-24, LTBEER24]",[Light Beer (24-pack)]
8,4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]",[AA Batteries 4-Pack]
10,aid first kit,2,"[FAID-01, FIRSTAID-KIT]","[First Aid Kit, First-Aid Kit]"
12,board game monopoly,3,"[BG-MONO, GAME-MONO, MONOPOLY-01]","[Board Game - Monopoly, Monopoly Board Game]"
13,bread white,2,"[BREAD-WH, WHITEBREAD]",[WHITE BREAD]
14,butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
15,candles emergency,2,"[CANDLE-EMRG, CNDL-EM]","[Candles (Emergency), Emergency Candles]"
16,canned soup,2,"[CNSOUP-01, SOUP-CAN]","[Canned Soup, Soup - Canned]"
17,cards playing,2,"[CARDS-01, PLAYCARD-STD]","[PLAYING CARDS, Playing Cards]"
20,chips pack variety,2,"[CHIPS-VP, CHIPVARIETY]","[Chips Variety Pack, Variety Pack - Chips]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
n_before = len(pd.read_csv(PATH))
assert cat['canonical_sku'].notna().all(), 'mapping created NaN canonical SKUs'
assert len(cat) == n_before, f'row count changed: {n_before} -> {len(cat)}'
assert cat['canonical_sku'].nunique() < cat['sku'].nunique(), 'mapping merged nothing'
print('mapping is safe: no nulls, row count unchanged at', len(cat))

mapping is safe: no nulls, row count unchanged at 45


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) two families beyond pop-tarts: monopoly (BG-MONO, GAME-MONO, MONOPOLY-01, names like “Monopoly Board Game” and “Board Game - Monopoly”) and first aid kits (FAID-01, FIRSTAID-KIT, “First Aid Kit” vs “First-Aid Kit”). i found them with the name-key grouping from Q5, checked each group by eye, then mapped each family to one canonical sku in the CANON dict.

b) consolidation stops being cleaning when the question becomes “is this the same product to a customer?” instead of “is this the same string?” one i’m not sure about is the pop-tarts: PT-12 is a 4-pack and POPTART-STRAW is a 6 pack, but i merged them since they’re both strawberry. a 4-pack and a 6-pack sell at different prices and customers might buy them differently. i’d ask the category manager or whoever owns the product catalog, since they decide how the company defines a product.